# edgar-extract — LoRA fine-tune (ücretsiz T4)

Adım ②. Ayrıntılı gerekçeler: repodaki `COLAB.md`.

**Runtime → Change runtime type → T4 GPU** seçili olmalı. Hücreleri sırayla çalıştırın.

İki yerde durup çıktıya bakmanız isteniyor (5. ve 6. hücre). Oralar, üç saatlik bir
koşuyu boşa harcamamak için var.

In [ ]:
# 1) GPU DOĞRULAMA
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# Beklenen: Tesla T4, 15360 MiB, 7.5
# 7.5 = Turing = bf16 YOK. train_lora.py bunu kendisi görüp fp16 seçiyor;
# sizin bir şey yapmanız gerekmiyor. Rehberlerden kopyaladığınız BAŞKA bir
# script bf16=True (TRL varsayılanı) ile burada patlarsa sebebi budur.

In [ ]:
# 2) KURULUM — sürümler SABİT
# TRL'in SFT API'si bu proje yazılırken değişti (max_seq_length -> max_length,
# varsayılan 1024). Eski adı veren script HATA VERMEZ, sessizce her örneği keser.
!pip install -q transformers==5.14.1 trl==1.9.2 peft==0.20.0 datasets==5.0.1 accelerate==1.14.0 bitsandbytes

# 🔴 torchao'yu KALDIRIN — kurmayı değil, KALDIRMAYI kastediyorum. ATLAMAYIN.
# Colab imajında torchao 0.10.0 HAZIR GELİYOR. peft'in LoRA dispatcher'ı her
# katman için is_torchao_available() çağırıyor ve eski sürümde bu fonksiyon
# False DÖNMÜYOR, ImportError ATIYOR — SFTTrainer daha kurulurken patlıyor.
# "torchao kullanmıyorum" sizi KORUMAZ. Yerelde torchao kurulu olmadığı için
# find_spec None döner, fonksiyon temizce False verir ve CPU smoke testi GEÇER:
# bu, yerelde ASLA göremeyeceğiniz bir arıza.
!pip uninstall -y -q torchao

In [ ]:
# 3) ORTAM + DEPO + VERİ — zip yüklemeye gerek YOK.
# Etiketler (160) ve span'ler 2026-08-02'de yayınlandı, yani eğitim seti depodan
# üretilebiliyor. Elle dosya yükleme adımı bu yüzden kaldırıldı: bir oturumun en
# kırılgan adımıydı.
import os, pathlib, json

KAGGLE = pathlib.Path('/kaggle').exists()
KOK    = '/kaggle/working/edgar-extract' if KAGGLE else '/content/edgar-extract'
print('ortam:', 'Kaggle' if KAGGLE else 'Colab', '| kök:', KOK)

# 🔴 KAGGLE: sağ panelde Settings -> Internet AÇIK olmalı. Kapalıysa bu hücre
#    git clone'da takılır ve sebebi hata mesajından anlaşılmaz. İnternet erişimi
#    telefon doğrulaması ister. Ayrıca Settings -> Accelerator -> GPU seçili olmalı.
if not pathlib.Path(KOK).exists():
    !git clone -q https://github.com/ozantosn24-ux/sec-filing-extraction-finetune.git {KOK}
os.chdir(KOK)

!python src/build_sft.py          # labels/ + spans/ -> sft_{train,dev,test}.jsonl
!python src/measure_tokens.py     # token_report.json — kesme korumasının tabanı

# Zincir TAM mı? Eksiği burada görmek, 30 dakikalık koşunun ortasında görmekten iyi.
for f in ['src/train_lora.py', 'src/predict.py',
          'data/processed/sft_train.jsonl', 'data/processed/sft_dev.jsonl',
          'data/processed/token_report.json']:
    print(('VAR   ' if pathlib.Path(f).exists() else 'EKSİK '), f)

r = json.load(open('data/processed/token_report.json'))
print('\nölçülen taban seq_len:', r['min_seq_len_no_truncation'],
      '-> önerilen', r['recommended_seq_len'])

---
## 🔴 DURAK 1 — smoke test

Aşağıdaki hücre öğrenmek için değil. Baktığınız **tek satır**:

```
kayip maskesi: ... token'in ...'sinde kayip hesaplaniyor (%4.7)
```

**~%5 olmalı.** Bu, kaybın yalnız JSON hedefinde hesaplandığı anlamına gelir — 700
token'lık talimat maskeleniyor. Oran yarıdan büyükse `completion_only_loss`
çalışmıyordur ve model **talimatı üretmeyi** öğrenir. O durumda devam etmeyin.

In [ ]:
# 4) SMOKE — 135M model, 2 adım, ~2 dakika
!python src/train_lora.py --smoke

---
## 🔴 DURAK 2 — VRAM probu

Gerçek modelle birkaç adım koşup **tepe VRAM**'i ölçer. OOM'u üç saatlik koşunun
40. adımında değil, burada görün.

`--probe` kayıtları **gerçek token uzunluğuna göre** sıralayıp en uzunları öne alır.
Bu ayrım önemli: dosyalar accession'a göre sıralı, yani ilk adımlar rastgele
uzunlukta — sıralamayan bir sonda **geçer**, sonra gerçek koşu en uzun diziye
geldiğinde patlar. Tepe VRAM en uzun diziyle oluşur, o yüzden prob onunla başlar.

Çıkan sayı bir **tavan**. %85'in altındaysa gerçek koşu rahat sığar. Üstündeyse
sırasıyla: `--rank 8` → `--4bit`.
**`--max-length`'i DÜŞÜRMEYİN** — 3072 ölçülmüş taban, altına inmek örnekleri keser
ve kesilen yer tam da çıkarılacak alanların bulunduğu bölgedir.

Prob adaptörü **kaydetmez**; sadece ölçer.

In [ ]:
# 5) VRAM PROBU — en uzun dizilerle, adaptör kaydedilmez
!python src/train_lora.py --probe

In [ ]:
# 6) GERÇEK KOŞU — 99 eğitim örneği
# ---------------------------------------------------------------- YAPILANDIRMA
EPOCHS = 5       # 3 = yayınlanmış koşu · 5 = schema/EPOCH_KARARI.md denemesi
TAG    = "e5"    # 3 epoch için "" bırakın
# -----------------------------------------------------------------------------
# 🔴 EFEMER DİZİNE YAZMAYIN. Colab'de /content, oturumla birlikte SİLİNİR ve
#    save_strategy="epoch" sizi KURTARMAZ — checkpoint'ler de oradadır. Bu proje
#    18 dakikalık bir eğitimi tam bu yüzden kaybetti. Drive maliyeti ÖLÇÜLDÜ: %6.
#    Kaggle'da /kaggle/working oturum çıktısı olarak KALICI, mount gerekmiyor.
ad = "lora-qwen2.5-1.5b" + (f"-{TAG}" if TAG else "")
if KAGGLE:
    OUT = f"/kaggle/working/{ad}"
else:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f"/content/drive/MyDrive/edgar-extract/{ad}"
print("adaptör dizini:", OUT, "| epoch:", EPOCHS)

# 🔴 EPOCHS'u TAG'i değiştirmeden artırmayın. 3 epoch adaptörü, yayınlanmış
#    sayıların (%61,1) arkasındaki TEK artifact — üzerine yazılırsa geri gelmez.
#
# 🔴 Bu bir DEVAM değil, YENİ koşudur. Öğrenme oranı zamanlayıcısı toplam adım
#    sayısına göre kurulur (linear decay, transformers varsayılanı), yani 5 epoch
#    koşusundaki checkpoint-39, 3 epoch koşusundakiyle AYNI MODEL DEĞİLDİR.
#    Soru "4. ve 5. epoch iyileştirdi mi" değil: "bu koşunun dev'deki en iyi
#    checkpoint'i 17/25'i geçiyor mu" (schema/EPOCH_KARARI.md).
#
# Başlangıçta basılan "optimizer adimi" satırı LoRA'nın kaç kez güncellendiğidir.
# Tahmin kabadır; 5 epoch için gerçek adım 65 (13 x 5) olmalı.
#
# 🔴 loss 'nan' olursa İLK ŞÜPHELİ fp16'dır (T4/P100'de bf16 yok), model ya da
#    veri değil. Veriyi kurcalamadan önce --lr 5e-5 deneyin.
#
# Diğer hiperparametreler DEĞİŞMEZ (rank 16, lr 1e-4, batch 1, grad-accum 8,
# max-length 3072, seed 42). İki şeyi aynı anda değiştirmek hangisinin etki
# ettiğini ölçülemez yapar.

!python src/train_lora.py --epochs {EPOCHS} --out {OUT}

---
## Sıra önemli: önce DEV, sonra TEST

`dev` (25 kayıt) model seçimi için — epoch/lr/checkpoint burada kıyaslanır.
`test` (36 kayıt) **bir kez** bakılır; şimdi bakılırsa dondurulmuş baseline sayısı
(%27,8) anlamını kaybeder.

Bu yüzden önce yalnız dev tahminleri üretilir, indirilir, **yerelde** ölçülür
(altın etiketler pakette yok — bilerek). Yapılandırma kesinleştikten sonra test.

In [ ]:
# 7) DEV tahminleri — HER EPOCH CHECKPOINT'I AYRI (model secimi)
#
# save_strategy="epoch" her epoch'un adaptorunu birakiyor; secim onlarin
# ARASINDA yapilir. Yalniz son adaptorle tahmin uretmek "kacinci epoch daha iyi"
# sorusunu sorulamaz hale getirir.
import glob, subprocess, os, pathlib

# 6. hucre kosulmadiysa (oturum koptu, yeniden baglandiniz) elle ayarlayin.
try:
    OUT, TAG, KAGGLE
except NameError:
    KAGGLE = pathlib.Path('/kaggle').exists()
    TAG = "e5"
    OUT = (f"/kaggle/working/lora-qwen2.5-1.5b-{TAG}" if KAGGLE
           else f"/content/drive/MyDrive/edgar-extract/lora-qwen2.5-1.5b-{TAG}")

# Dosya adina TAG giriyor: aksi halde 5 epoch kosusunun preds_ft_dev_13.jsonl'i
# 3 epoch kosusununkinin UZERINE yazar ve yayinlanmis secim kaydi kaybolur.
pre = f"preds_ft_{TAG}_dev" if TAG else "preds_ft_dev"

ckpts = sorted(glob.glob(f"{OUT}/checkpoint-*"), key=lambda s: int(s.split("-")[-1]))
print("adaptor dizini:", OUT)
print("bulunan checkpoint:", [os.path.basename(c) for c in ckpts] or "YOK")
assert ckpts, f"{OUT} altinda checkpoint yok — 6. hucre kosuldu mu?"

for ck in ckpts:
    n = ck.split("-")[-1]
    subprocess.run(["python", "src/predict.py", "--adapter", ck, "--split", "dev",
                    "-o", f"data/processed/{pre}_{n}.jsonl"], check=True)

# TEST'e BU KOSUDA dokunulmuyor. Karar kurali (schema/EPOCH_KARARI.md) once
# dev'de uygulanir; test ancak yapilandirma kesinlesirse ve "IKINCI olcum"
# etiketiyle kosulur.
if KAGGLE:
    # /kaggle/working oturum ciktisi olarak kalir; sag panel > Output'tan inen zip.
    !cp data/processed/{pre}_*.jsonl /kaggle/working/
    print("\ndev tahminleri /kaggle/working altina kopyalandi — sag panel > Output.")
    !ls -la /kaggle/working/{pre}_*.jsonl
else:
    !zip -qr dev_preds_{TAG}.zip data/processed/{pre}_*.jsonl
    from google.colab import files
    files.download(f'dev_preds_{TAG}.zip')

In [ ]:
# 8) TEST tahminleri — DUR. Bu hucre kosulmadan once schema/EPOCH_KARARI.md okunur.
#
# 🔴 TEST'E BIR KEZ BAKILDI (2026-08-01). Bu hucreyi yeniden kosmak IKINCI
#    olcumdur ve raporda oyle etiketlenmek ZORUNDADIR. Yapilandirma dev'de
#    kesinlesmeden burayi calistirmayin: kesinlesmemis bir yapilandirmayla test'e
#    bakmak, test setini validation'a cevirir ve %27,8 baseline'i anlamsizlastirir.
KOSMAYA_HAZIR = False   # bilinerek True yapin
assert KOSMAYA_HAZIR, "schema/EPOCH_KARARI.md — karar dev'de verilmeden test kosulmaz"

SECILEN = ""   # dev'de en iyi cikan checkpoint; bos = son adaptor
adapter = SECILEN or OUT
print("kullanilan adaptor:", adapter)

!python src/predict.py --adapter {adapter} --split test -o data/processed/preds_ft_test.jsonl

# 🔴 AYNI modelin ADAPTORSUZ hali. Bu kosu opsiyonel DEGIL: fine-tuned skor tek
# basina "LoRA mi ogretti, taban model zaten biliyor muydu" sorusunu AYIRT ETMEZ.
!python src/predict.py --split test -o data/processed/preds_base1p5b_test.jsonl

# Dorduncu yarismaci: prompted BUYUK model, 4-bit.
# 🔴 7B DEGIL — OLCULDU: --4bit modeli 4-bit YUKLER ama once fp16 agirliklari
#    INDIRMEK zorunda (Qwen2.5-7B icin 15,2 GB) ve bu ucretsiz oturumu %32'de
#    OLDURDU, /content ile birlikte adaptoru ve tum tahminleri goturdu.
#    Darbogaz VRAM degil, indirme/disk. 3B (~6 GB) sigdi ve olculen yarismaci bu.
!python src/predict.py --model Qwen/Qwen2.5-3B-Instruct --4bit --split test -o data/processed/preds_prompted3b_test.jsonl

for f in ['preds_ft_test.jsonl', 'preds_base1p5b_test.jsonl', 'preds_prompted3b_test.jsonl']:
    if KAGGLE:
        !cp data/processed/{f} /kaggle/working/
    else:
        from google.colab import files
        files.download(f'data/processed/{f}')
print("tahminler hazir — olcum YERELDE: altin etiketler depoda, evaluate.py ile.")

In [ ]:
# 9) ADAPTÖRÜ İNDİRİN — artifact'ın kendisi, birkaç MB
# Kaggle'da OUT zaten /kaggle/working altında, yani oturum çıktısında duruyor.
# Colab'de Drive'a yazıldı; yine de yerel bir kopya isterseniz:
if KAGGLE:
    !ls -la {OUT}
    print("\nadaptör /kaggle/working altında — sağ panel > Output'tan inebilir.")
else:
    !zip -qr adapter_{TAG}.zip {OUT}
    from google.colab import files
    files.download(f'adapter_{TAG}.zip')

---
## Sonra: yerelde ölçüm

İndirilen dosyaları `data/processed/` altına koyup:

```bash
python src/evaluate.py data/processed/preds_ft_dev_*.jsonl --split dev    # seçim: en iyi epoch

python src/evaluate.py \
  data/processed/preds_regex_test.jsonl \
  data/processed/preds_ft_test.jsonl \
  data/processed/preds_prompted7b_test.jsonl \
  --json-out data/processed/eval_all.json
```

**Aşılması gereken çubuk** (kural-tabanlı, test): tam kayıt **%27,8** · zor vaka
**%81,6** · doğru abstention %90,2 · şema geçerliliği %100.

⚠️ Regex'in **dev** skoru (%68,0) kirlidir — kurallar dev oyulmadan önce tüm train
üzerinde ayarlandı. Fine-tuned modelin dev skoruyla karşılaştırmayın; karşılaştırma
yeri yalnız test.

Fine-tune bunları geçmezse sonuç **"fine-tune bu görevde kural-tabanlıyı yenmedi"**
olur ve öyle raporlanır. Ölçümün amacı kazanmak değil, öğrenmek.